# 把网页发布到公网

## 当前状态与部署目标

开始前应该已经具备：

- **本地电脑**：`~/zero-to-tech/` 中有 `index.html`、`style.css` 和 `script.js`。
- **GitHub**：本地项目已经推送到 `zero-to-tech` 远程仓库。
- **云服务器**：Ubuntu 已安装 Nginx，浏览器访问公网 IP 能看到默认欢迎页。

本节要接通下面的完整链路：

```text
本地电脑                         GitHub
index.html  ── commit + push ──→  zero-to-tech 仓库
style.css                              │
script.js                              │ clone / pull
                                       ↓
浏览器  ←── 80 端口 / Nginx ──  云服务器 ~/zero-to-tech
```

核心任务只有两个：

1. 让云服务器从 GitHub 获取项目代码。
2. 让 Nginx 从新的项目目录读取网页文件。

## 第一步：登录云服务器

在本地电脑的终端中执行：

```bash
ssh ubuntu@你的公网IP
```

登录成功后，提示符会变成类似：

```text
ubuntu@your-server:~$
```

从这里开始，除非特别标明“本地电脑”，命令都在云服务器的 SSH 会话中执行。执行命令前可以观察提示符，避免在错误的机器上操作。

## 第二步：确认服务器已经安装 Git

```bash
git --version
```

Ubuntu 云镜像通常已经包含 Git。看到 `git version 2.x.x` 即可继续。若提示找不到命令，再执行：

```bash
sudo apt update
sudo apt install -y git
git --version
```

`apt update` 刷新软件包信息，`apt install -y git` 安装 Git，`-y` 表示自动确认安装提示。

## 第三步：让服务器获得 GitHub SSH 身份

每台机器都应该拥有自己的 SSH 密钥。不能把本地电脑的私钥复制到服务器；云服务器需要单独生成一对密钥，再把它的公钥添加到 GitHub。

### 在服务器上生成密钥

```bash
ssh-keygen -t ed25519 -C "你的邮箱"
```

初次配置可以连续按回车使用默认值。生成的两个文件为：

| 文件 | 含义 |
|------|------|
| `~/.ssh/id_ed25519` | 私钥，只能保留在当前服务器 |
| `~/.ssh/id_ed25519.pub` | 公钥，可以添加到 GitHub |

### 查看公钥

```bash
cat ~/.ssh/id_ed25519.pub
```

复制以 `ssh-ed25519` 开头的完整一行，然后进入：

```text
GitHub → Settings → SSH and GPG keys → New SSH key
```

标题应能区分设备，例如“云服务器-阿里云”。本地电脑原有的公钥可以继续保留。

### 验证连接

```bash
ssh -T git@github.com
```

第一次连接时输入 `yes`。看到 `successfully authenticated` 表示服务器已经可以通过 SSH 访问 GitHub。

> Public 仓库也可以直接通过 HTTPS clone；Private 仓库使用 SSH 更方便。本课程统一使用 SSH。

## 第四步：把项目 clone 到服务器

进入服务器当前用户的家目录并克隆仓库：

```bash
cd ~
git clone git@github.com:你的用户名/zero-to-tech.git
```

这里不需要 `sudo`，因为 `/home/ubuntu` 是 `ubuntu` 用户自己的家目录。克隆完成后检查文件：

```bash
ls ~/zero-to-tech
```

应该能看到：

```text
index.html  style.css  script.js
```

本地电脑和服务器都可以使用 `~/zero-to-tech`，但 `~` 的真实位置不同：

| 机器 | `~` 通常代表 |
|------|------|
| macOS 本地电脑 | `/Users/你的用户名` |
| Ubuntu 云服务器 | `/home/ubuntu` |

> `clone` 用于第一次把完整仓库下载到服务器；以后更新已有仓库使用 `pull`。

## 第五步：看懂 Nginx 配置结构

进入 Nginx 配置目录：

```bash
cd /etc/nginx
ls
```

Ubuntu / Debian 会把 Nginx 配置拆成多个文件和目录。真正的总入口是 `/etc/nginx/nginx.conf`：

```bash
cat /etc/nginx/nginx.conf
```

Nginx 启动时直接读取 `nginx.conf`，再通过 `include` 加载其他配置：

```text
nginx.conf
├── 全局配置：user、worker_processes、pid、error_log
├── events { ... }：连接与并发参数
└── http { ... }：HTTP 服务配置
    ├── include /etc/nginx/mime.types
    ├── include /etc/nginx/conf.d/*.conf
    └── include /etc/nginx/sites-enabled/*
```

这与 `index.html` 通过 `<link>` 和 `<script>` 引入 CSS、JavaScript 的方式相似：一个主入口把多个独立文件组织起来。

| 名称 | 作用 |
|------|------|
| `nginx.conf` | Nginx 配置总入口 |
| `sites-available/` | 保存已经写好的网站配置 |
| `sites-enabled/` | 保存当前启用的网站配置链接 |
| `conf.d/` | 额外的全局配置片段 |
| `snippets/` | 可复用配置片段 |
| `mime.types` | 文件扩展名到 MIME 类型的映射 |
| `modules-available/`、`modules-enabled/` | 可用与已启用的动态模块 |

其他代理参数和历史字符编码文件暂时不需要修改。不同 Linux 发行版的目录结构可能不同，本节说明的是 Ubuntu / Debian 的默认组织方式。

## sites-available 与 sites-enabled

这两个目录可以理解为“可用配置”和“启用配置”：

```text
/etc/nginx/sites-available/   保存所有网站配置
/etc/nginx/sites-enabled/     保存当前启用的配置链接
```

查看链接关系：

```bash
ls -l /etc/nginx/sites-enabled/
```

默认结果通常类似：

```text
default -> /etc/nginx/sites-available/default
```

`->` 表示 `sites-enabled/default` 是指向 `sites-available/default` 的**软链接（symlink）**，类似快捷方式。因此修改任意一个路径，实际修改的是同一份配置。

这种设计允许保留网站配置，只通过添加或删除 `sites-enabled` 中的链接控制是否启用。

## 看懂默认 server 配置

查看当前启用的默认网站：

```bash
cat /etc/nginx/sites-enabled/default
```

忽略以 `#` 开头的注释后，核心结构类似：

```nginx
server {
    listen 80 default_server;
    listen [::]:80 default_server;

    root /var/www/html;
    index index.html index.htm index.nginx-debian.html;

    server_name _;

    location / {
        try_files $uri $uri/ =404;
    }
}
```

| 配置 | 含义 |
|------|------|
| `server { ... }` | 定义一个网站 |
| `listen 80` | 监听 HTTP 默认端口 80 |
| `default_server` | 没有其他网站匹配时使用当前配置 |
| `root /var/www/html` | 从该目录寻找要返回的文件 |
| `index ...` | 访问目录时按顺序寻找默认入口文件 |
| `server_name _` | 暂时接收未匹配其他域名的请求 |
| `location /` | 为所有 URL 路径定义处理规则 |
| `try_files $uri $uri/ =404` | 依次尝试文件、目录，均不存在则返回 404 |

例如访问 `/about.html` 时，Nginx 会在 `root` 后拼接路径，默认寻找 `/var/www/html/about.html`。访问 `/` 时会按 `index` 列表寻找入口文件。

## 第六步：让 Nginx 指向项目目录

用管理员权限编辑默认站点配置：

```bash
sudo vim /etc/nginx/sites-enabled/default
```

在 Vim 中：

1. 按 `/`，输入 `root /var` 并回车，定位原配置。
2. 按 `i` 进入插入模式。
3. 把 `/var/www/html` 改为 `/home/ubuntu/zero-to-tech`。
4. 确认行末仍然有分号 `;`。
5. 按 `Esc`，输入 `:wq` 并回车。

修改后的关键配置应为：

```nginx
root /home/ubuntu/zero-to-tech;
```

必须使用绝对路径，不能在 Nginx 配置中写 `~/zero-to-tech`。`~` 是 Shell 展开的简写，Nginx 不会把它自动解释为 `/home/ubuntu`。

## 第七步：检查配置并重新加载 Nginx

先进行语法检查：

```bash
sudo nginx -t
```

正常结果包含：

```text
syntax is ok
test is successful
```

如果检查失败，错误信息会指出文件和行号。修正后重新检查，**校验通过前不要 reload**。

检查通过后平滑加载新配置：

```bash
sudo systemctl reload nginx
```

| 操作 | 含义 |
|------|------|
| `reload` | 重新读取配置，尽量不中断现有连接 |
| `restart` | 完全停止并重新启动进程 |

只修改配置时使用 `reload` 即可。命令没有输出通常表示执行成功。

## 第八步：理解并修复 404 或 403

浏览器访问下面的地址：

```text
http://你的公网IP
```

此时可能出现 `404 Not Found` 或 `403 Forbidden`。配置和文件路径可能都正确，真正原因是 Linux 目录权限。

Nginx 默认以 `www-data` 用户身份读取文件。它要访问：

```text
/ → home → ubuntu → zero-to-tech → index.html
```

必须拥有穿过路径中每一级目录的执行权限。检查家目录：

```bash
ls -ld /home/ubuntu
```

常见结果类似：

```text
drwxr-x--- 5 ubuntu ubuntu ... /home/ubuntu
```

最后三位 `---` 表示其他用户没有权限，`www-data` 因而不能穿过 `/home/ubuntu`。使用最小授权修复：

```bash
sudo chmod o+x /home/ubuntu
```

- `o`：others，既不是属主也不在属主组中的用户。
- `+x`：给目录增加穿过权限。
- 没有增加 `r`：其他用户仍不能列出家目录中的全部内容。

再次检查时，权限通常会变成 `drwxr-x--x`。刷新浏览器后，应能看到自己的卡片页面并使用按钮。这个权限修改通常只需执行一次。

## 第九步：完成一次公网更新

### 在本地电脑修改并推送

把 `index.html` 中的标题从：

```html
<h1>你好，互联网</h1>
```

改为：

```html
<h1>你好，世界</h1>
```

保存后，在**本地终端**执行：

```bash
cd ~/zero-to-tech
git add .
git commit -m "改一下标题"
git push
```

此时 GitHub 已经更新，但云服务器仍保留旧版本。

### 在服务器拉取更新

回到云服务器的 SSH 会话：

```bash
cd ~/zero-to-tech
git pull
```

最后刷新浏览器，标题应变为“你好，世界”。静态 HTML、CSS 或 JavaScript 文件改变后，不需要 reload Nginx；Nginx 会在新请求中直接读取更新后的文件。

## 部署后的标准更新流程

以后每次让本地改动在公网生效，都经过三段操作：

### 1. 本地电脑：提交并推送

```bash
cd ~/zero-to-tech
git add .
git commit -m "写清楚这次改了什么"
git push
```

### 2. 云服务器：拉取最新提交

```bash
cd ~/zero-to-tech
git pull
```

### 3. 浏览器：刷新页面

```text
本地修改 → push 到 GitHub → 服务器 pull → 浏览器刷新
```

这是最朴素的手动部署流程。未来的自动部署只是让服务器在 GitHub 更新后自动执行同类操作，底层链路并没有改变。

> 服务器上的项目目录应当作为 GitHub 的部署副本。代码修改放在本地完成，不要直接编辑服务器中的文件。

## 常见问题排查

| 问题 | 优先检查 |
|------|------|
| 仍显示 Nginx 欢迎页 | `root` 是否修改正确、是否通过 `nginx -t`、是否执行 reload；尝试强制刷新或无痕窗口 |
| `nginx -t` 报错 | 根据文件和行号检查路径、括号及行尾分号，修复前不要 reload |
| 页面显示 404 / 403 | `root` 路径、文件是否存在，以及 `/home/ubuntu` 是否有 `o+x` 权限 |
| `git clone` 提示 publickey | 服务器公钥是否添加到 GitHub，`ssh -T git@github.com` 是否成功 |
| `git pull` 提示有本地改动 | 服务器文件被手动修改，应停止在服务器直接编辑代码 |
| push 后服务器页面不变 | 是否已经在服务器项目目录执行 `git pull` |
| pull 后浏览器仍是旧页面 | 强制刷新或使用无痕窗口排除浏览器缓存 |

如果服务器目录已经出现不需要保留的手动修改，可先查看：

```bash
cd ~/zero-to-tech
git status
```

课程给出的恢复方式是丢弃所有未提交改动后重新拉取：

```bash
git checkout .
git pull
```

> `git checkout .` 会丢弃服务器项目中的未提交修改，无法从 Git 恢复。执行前必须确认这些改动确实不需要保留。

## 本节命令速查

```bash
# 本地：登录服务器
ssh ubuntu@你的公网IP

# 服务器：准备 GitHub SSH 与项目
git --version
ssh-keygen -t ed25519 -C "你的邮箱"
cat ~/.ssh/id_ed25519.pub
ssh -T git@github.com
cd ~
git clone git@github.com:你的用户名/zero-to-tech.git

# 服务器：查看并修改 Nginx
cat /etc/nginx/nginx.conf
ls -l /etc/nginx/sites-enabled/
cat /etc/nginx/sites-enabled/default
sudo vim /etc/nginx/sites-enabled/default
sudo nginx -t
sudo systemctl reload nginx

# 服务器：允许 Nginx 穿过用户家目录
ls -ld /home/ubuntu
sudo chmod o+x /home/ubuntu

# 服务器：更新已部署代码
cd ~/zero-to-tech
git pull
```

完成本节后，应该能解释 GitHub、云服务器、Nginx 和浏览器各自的作用，并独立走通“本地修改 → push → 服务器 pull → 浏览器刷新”的上线流程。

## 模块 3 的完整成果

到这里已经走通：

```text
本地开发
  → HTML / CSS / JavaScript 分工
  → Git 本地版本管理
  → GitHub 远程托管
  → 云服务器部署
  → Nginx 通过 80 端口提供网页
  → 互联网用户通过公网 IP 访问
```

页面本身虽然简单，但这条从开发到公网访问的完整链路，是后续构建更复杂产品的基础。